# VQA Yes/No — Colab Runbook

Trains the attention-based yes/no VQA model on VQA v2 (COCO images + VQA annotations).

**Before running:** `Runtime → Change runtime type → T4 GPU`

**Steps:**
1. Clone repo (branch `experiment/visualQA`)
2. Install dependencies
3. Mount Drive (for persisting checkpoints)
4. Set up Kaggle credentials
5. Download VQA v2 annotations + COCO images via Kaggle API
6. Build question vocabulary
7. Train
8. Evaluate accuracy
9. Run inference on a single image

## 1. Clone Repo

In [ ]:
import os
import subprocess
from pathlib import Path

GITHUB_TOKEN = ""  # <-- paste your GitHub personal access token here

REPO_URL    = f"https://{GITHUB_TOKEN}@github.com/Miwi343/Neural_Image_Caption_Generator.git"
REPO_BRANCH = "experiment/visualQA"
REPO_DIR    = Path("/content/Neural_Image_Caption_Generator")

if not (REPO_DIR / ".git").exists():
    subprocess.run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "remote", "set-url", "origin", REPO_URL], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)

os.chdir(REPO_DIR)
print("Branch:", subprocess.check_output(["git", "branch", "--show-current"]).decode().strip())

## 2. Install Dependencies

In [ ]:
# Colab already has torch/torchvision; just install anything extra from requirements.txt
!pip install -q -r requirements.txt
!pip install -q kaggle

In [ ]:
import torch

print("torch       :", torch.__version__)
print("cuda        :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu         :", torch.cuda.get_device_name(0))
    print("gpu memory  :", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

## 3. Mount Drive

Checkpoints and results will be written to Drive so they survive Colab disconnects.
Edit `DRIVE_OUTPUT_DIR` if you want them somewhere else.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import shutil

DRIVE_OUTPUT_DIR = Path("/content/drive/MyDrive/vqa_results")
DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def link_output_dir(local_name: str):
    """Symlink a local output dir into Drive so writes persist."""
    local  = Path(local_name)
    remote = DRIVE_OUTPUT_DIR / local_name
    remote.mkdir(parents=True, exist_ok=True)

    if local.is_symlink():
        print(f"Already linked: {local} -> {local.resolve()}")
        return
    if local.exists():
        contents = list(local.iterdir())
        if len(contents) == 0 or (len(contents) == 1 and contents[0].name == ".gitkeep"):
            shutil.rmtree(local)
        else:
            print(f"Using existing local {local}; not replacing.")
            return
    os.symlink(remote, local)
    print(f"Linked: {local} -> {remote}")

link_output_dir("checkpoints_vqa")
link_output_dir("results_vqa")

## 4. Kaggle Credentials

Kaggle now uses a plain API token (no `kaggle.json`).

**To get your credentials:**
1. Go to [kaggle.com](https://www.kaggle.com) → profile picture (top right) → **Settings**
2. Scroll to **API** → click **Create New Token**
3. Copy the token string shown on screen — you only see it once
4. Your username is shown at the top of the Settings page

**Option A (recommended):** Save to Drive once, reuse every session — run the cell below, filling in your username + token. The credentials file gets written to Drive and reloaded automatically on future runs.

**Option B:** Paste directly into the environment variables cell — credentials live only in the runtime and are never saved anywhere.

In [ ]:
# --- Option A: save credentials to Drive (recommended) ---
# Fill in your username and token, run once, then this cell
# will reload them automatically on future sessions.

import json, stat, shutil
from pathlib import Path

KAGGLE_USERNAME = "YOUR_KAGGLE_USERNAME"   # <-- replace
KAGGLE_TOKEN    = "YOUR_KAGGLE_API_TOKEN"  # <-- replace (the long string from Settings)

KAGGLE_JSON_DRIVE = Path("/content/drive/MyDrive/kaggle_credentials.json")
KAGGLE_CONFIG_DIR = Path.home() / ".config" / "kaggle"
KAGGLE_CONFIG_DIR.mkdir(parents=True, exist_ok=True)

if KAGGLE_USERNAME != "YOUR_KAGGLE_USERNAME":
    # Save to Drive so future sessions can skip filling this in
    creds = {"username": KAGGLE_USERNAME, "key": KAGGLE_TOKEN}
    KAGGLE_JSON_DRIVE.write_text(json.dumps(creds))
    print(f"Saved credentials to Drive: {KAGGLE_JSON_DRIVE}")
elif KAGGLE_JSON_DRIVE.exists():
    # Already saved from a previous session — load from Drive
    creds = json.loads(KAGGLE_JSON_DRIVE.read_text())
    KAGGLE_USERNAME = creds["username"]
    KAGGLE_TOKEN    = creds["key"]
    print(f"Loaded credentials from Drive for user: {KAGGLE_USERNAME}")
else:
    print("No credentials found — fill in KAGGLE_USERNAME and KAGGLE_TOKEN above, or use Option B.")

# Write to the location the kaggle CLI expects
dest = KAGGLE_CONFIG_DIR / "kaggle.json"
dest.write_text(json.dumps({"username": KAGGLE_USERNAME, "key": KAGGLE_TOKEN}))
dest.chmod(stat.S_IRUSR | stat.S_IWUSR)  # chmod 600 — required by kaggle CLI
print(f"Credentials written to {dest}")

In [ ]:
# --- Option B: paste credentials directly (not saved anywhere) ---
# Only run this cell if Option A didn't work.

import os
os.environ["KAGGLE_USERNAME"] = "YOUR_KAGGLE_USERNAME"   # <-- replace
os.environ["KAGGLE_KEY"]      = "YOUR_KAGGLE_API_TOKEN"  # <-- replace
print("Kaggle credentials set via environment variables (this session only).")

## 5. Download Data

- **VQA v2 annotation + question JSONs** — downloaded from [visualqa.org](https://visualqa.org/download.html), no Kaggle needed (~80 MB)
- **COCO images** — only the images that actually appear in yes/no questions are downloaded (~8 GB instead of 19 GB), directly from the official COCO S3 bucket in parallel

In [ ]:
DATA_ROOT = Path("data/vqa")

In [ ]:
import subprocess, zipfile

DATA_ROOT.mkdir(parents=True, exist_ok=True)

# Official VQA v2 download URLs from visualqa.org — no Kaggle account needed
VQA_DOWNLOADS = [
    (
        "https://s3.amazonaws.com/cvmlp/vqa/mscoco/vqa/v2_Annotations_Train_mscoco.zip",
        "v2_mscoco_train2014_annotations.json",
    ),
    (
        "https://s3.amazonaws.com/cvmlp/vqa/mscoco/vqa/v2_Annotations_Val_mscoco.zip",
        "v2_mscoco_val2014_annotations.json",
    ),
    (
        "https://s3.amazonaws.com/cvmlp/vqa/mscoco/vqa/v2_Questions_Train_mscoco.zip",
        "v2_OpenEnded_mscoco_train2014_questions.json",
    ),
    (
        "https://s3.amazonaws.com/cvmlp/vqa/mscoco/vqa/v2_Questions_Val_mscoco.zip",
        "v2_OpenEnded_mscoco_val2014_questions.json",
    ),
]

print("Downloading VQA v2 annotation + question JSONs from visualqa.org …")
for url, expected_json in VQA_DOWNLOADS:
    json_dest = DATA_ROOT / expected_json
    if json_dest.exists():
        print(f"  Already exists: {expected_json}")
        continue

    zip_name = url.split("/")[-1]
    zip_dest = DATA_ROOT / zip_name
    print(f"  Downloading {zip_name} …")
    subprocess.run(["wget", "-q", "-O", str(zip_dest), url], check=True)

    print(f"  Extracting {zip_name} …")
    with zipfile.ZipFile(zip_dest) as z:
        z.extractall(DATA_ROOT)
    zip_dest.unlink()
    print(f"  Done → {expected_json}")

print("\nAnnotation files:")
for f in sorted(DATA_ROOT.glob("*.json")):
    print(f"  {f.name}  ({f.stat().st_size // 1_000_000} MB)")

In [ ]:
import json
import urllib.request
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

IMG_DIR = DATA_ROOT / "images"
(IMG_DIR / "train2014").mkdir(parents=True, exist_ok=True)
(IMG_DIR / "val2014").mkdir(parents=True, exist_ok=True)

def get_needed_image_ids(data_root: Path):
    """Read annotation JSONs and return only image IDs that have yes/no questions."""
    needed = {"train2014": set(), "val2014": set()}
    for split, coco_split in [("train", "train2014"), ("val", "val2014")]:
        ann_path = data_root / f"v2_mscoco_{coco_split}_annotations.json"
        with open(ann_path) as f:
            for ann in json.load(f)["annotations"]:
                if ann["answer_type"] == "yes/no":
                    needed[coco_split].add(ann["image_id"])
    return needed

def download_image(image_id: int, coco_split: str, img_dir: Path) -> str:
    filename = f"COCO_{coco_split}_{image_id:012d}.jpg"
    dest = img_dir / coco_split / filename
    if dest.exists():
        return f"skip:{filename}"
    url = f"http://images.cocodataset.org/{coco_split}/{filename}"
    try:
        urllib.request.urlretrieve(url, dest)
        return f"ok:{filename}"
    except Exception as e:
        return f"err:{filename}:{e}"

print("Finding yes/no image IDs from annotation JSONs …")
needed = get_needed_image_ids(DATA_ROOT)
for split, ids in needed.items():
    print(f"  {split}: {len(ids):,} unique images needed")

total = sum(len(v) for v in needed.values())
print(f"\nDownloading {total:,} images (parallel, 16 threads) …")
print("This may take 20-40 minutes depending on connection speed.\n")

done = 0
errors = []
with ThreadPoolExecutor(max_workers=16) as pool:
    futures = {
        pool.submit(download_image, img_id, coco_split, IMG_DIR): (img_id, coco_split)
        for coco_split, ids in needed.items()
        for img_id in ids
    }
    for future in as_completed(futures):
        result = future.result()
        done += 1
        if result.startswith("err:"):
            errors.append(result)
        if done % 1000 == 0:
            print(f"  {done}/{total} ({done/total:.0%}) — {len(errors)} errors so far")

print(f"\nDone. {done - len(errors):,} downloaded, {len(errors)} errors.")
if errors:
    print("First 5 errors:", errors[:5])

## 6. Build Vocabulary

Tokenises all training questions and keeps the top 15,000 words.
Saved to `data/vqa/vocab.json` — only needs to run once.

In [ ]:
from vqa.dataset import QuestionVocabulary, build_and_save_vocab
from vqa.config import VOCAB_PATH, VOCAB_SIZE

vocab_path = Path(VOCAB_PATH)

if vocab_path.exists():
    vocab = QuestionVocabulary.load(str(vocab_path))
    print(f"Loaded existing vocabulary: {len(vocab):,} tokens")
else:
    print("Building vocabulary from training questions …")
    vocab = build_and_save_vocab(
        data_root=str(DATA_ROOT),
        vocab_path=str(vocab_path),
        max_size=VOCAB_SIZE,
    )

print(f"Vocabulary size: {len(vocab):,}")
print("Sample words:", list(vocab.word2idx.keys())[2:12])

### Quick dataset sanity check

Loads the first few samples and prints them so you can confirm the data pipeline works before spending time training.

In [ ]:
import json
from vqa.dataset import VQAYesNoDataset

ds = VQAYesNoDataset(
    data_root=str(DATA_ROOT),
    vocab=vocab,
    split="val",
    max_samples=2000,   # small subset for the check
)

print(f"Val samples (yes/no only): {len(ds):,}")

# Label balance
labels = [ds.pairs[i][2] for i in range(len(ds))]
yes_count = sum(labels)
print(f"Yes: {yes_count:,}  |  No: {len(labels)-yes_count:,}  (balance: {yes_count/len(labels):.2%} yes)")

# Show 5 examples
print("\nSample questions:")
for i in range(5):
    img_id, q, label = ds.pairs[i]
    print(f"  [{i}] image_id={img_id}  q={q!r}  label={'yes' if label else 'no'}")

## 7. Train

Runs `vqa/train.py` as a module.  Checkpoints go to `checkpoints_vqa/` (symlinked to Drive).

Key settings in `vqa/config.py`:
- Encoder frozen for first 5 epochs, then top VGG layers unfreeze at `ENCODER_LR=1e-4`
- Adam, `lr=3e-4`, `ReduceLROnPlateau` on val accuracy
- Early stopping after 7 non-improving epochs

In [ ]:
# Optional: reduce dataset size for a quick smoke-test run.
# Set MAX_SAMPLES=None to use the full dataset.
MAX_SAMPLES = None   # e.g. 10000 for a fast test

if MAX_SAMPLES is not None:
    import vqa.config as cfg
    # Patch get_vqa_dataloader calls in train.py to cap samples
    # The cleaner way is to pass --max_samples, but we do it via env var here
    os.environ["VQA_MAX_SAMPLES"] = str(MAX_SAMPLES)
    print(f"Capped dataset to {MAX_SAMPLES} samples for smoke test.")
else:
    os.environ.pop("VQA_MAX_SAMPLES", None)
    print("Using full dataset.")

In [ ]:
!python -m vqa.train

In [ ]:
# Plot training curves
import csv
import matplotlib.pyplot as plt

log_path = Path("results_vqa/training_log.csv")
if log_path.exists():
    with open(log_path) as f:
        rows = list(csv.DictReader(f))

    epochs    = [int(r["epoch"])          for r in rows]
    train_acc = [float(r["train_acc"])    for r in rows]
    val_acc   = [float(r["val_acc"])      for r in rows]
    train_loss = [float(r["train_loss"])  for r in rows]
    val_loss   = [float(r["val_loss"])    for r in rows]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    ax1.plot(epochs, train_acc, label="train")
    ax1.plot(epochs, val_acc,   label="val")
    ax1.set_title("Accuracy")
    ax1.set_xlabel("epoch")
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    ax2.plot(epochs, train_loss, label="train")
    ax2.plot(epochs, val_loss,   label="val")
    ax2.set_title("Loss")
    ax2.set_xlabel("epoch")
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig("results_vqa/training_curves.png", dpi=150)
    plt.show()
else:
    print("No training log yet — run the training cell first.")

## 8. Evaluate

Reports overall accuracy plus per-class (yes / no) breakdown.

In [ ]:
!python -m vqa.evaluate \
    --checkpoint checkpoints_vqa/best.pt \
    --data_root data/vqa \
    --vocab_path data/vqa/vocab.json \
    --batch_size 64

## 9. Single-Image Inference

Upload any image and ask a yes/no question about it.  The cell below also visualises where the model attended when forming its answer.

In [ ]:
# Upload an image directly in Colab
from google.colab import files
uploaded = files.upload()
IMAGE_PATH = next(iter(uploaded))   # path of the uploaded file
print("Using image:", IMAGE_PATH)

In [ ]:
# Or just hardcode a path if you already have one
# IMAGE_PATH = "data/vqa/images/val2014/COCO_val2014_000000000042.jpg"

QUESTION = "Is there a dog in this image?"   # <-- change this

from vqa.evaluate import answer_question

result = answer_question(
    checkpoint_path="checkpoints_vqa/best.pt",
    image_path=IMAGE_PATH,
    question=QUESTION,
    vocab_path="data/vqa/vocab.json",
)
print(f"\nAnswer     : {result['answer'].upper()}")
print(f"Confidence : {result['confidence']:.1%}  (P(yes))")

In [ ]:
# Visualise the attention map overlaid on the image
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from scipy.ndimage import gaussian_filter

img = np.array(Image.open(IMAGE_PATH).convert("RGB").resize((224, 224)))

# alpha is (196,) — reshape to 14x14, upsample to 224x224 (same as visualize.py)
alpha = result["alpha"].reshape(14, 14)
alpha_up = np.kron(alpha, np.ones((16, 16)))   # nearest-neighbour 16x upsample
alpha_smooth = gaussian_filter(alpha_up, sigma=8)
alpha_norm = (alpha_smooth - alpha_smooth.min()) / (alpha_smooth.max() - alpha_smooth.min() + 1e-8)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(img)
axes[0].set_title("Input image")
axes[0].axis("off")

axes[1].imshow(img)
axes[1].imshow(alpha_norm, alpha=0.55, cmap="hot")
answer_str = result['answer'].upper()
conf_str   = f"{result['confidence']:.1%}"
axes[1].set_title(f"Attention  |  A: {answer_str} ({conf_str})")
axes[1].axis("off")

fig.suptitle(f"Q: {QUESTION}", fontsize=12)
plt.tight_layout()
plt.savefig("results_vqa/inference_attention.png", dpi=150, bbox_inches="tight")
plt.show()

## 10. Output locations

Everything important is persisted to Drive under `vqa_results/`:

```
MyDrive/vqa_results/
  checkpoints_vqa/
    best.pt          ← best checkpoint by val accuracy
    epoch_NNN.pt     ← per-epoch checkpoints
  results_vqa/
    training_log.csv
    training_curves.png
    inference_attention.png
```

In [ ]:
# Back up vocab.json to Drive too
!mkdir -p "/content/drive/MyDrive/vqa_results/data/vqa"
!cp data/vqa/vocab.json "/content/drive/MyDrive/vqa_results/data/vqa/vocab.json"

!find "/content/drive/MyDrive/vqa_results" -type f | sort